# Run PG

Run this notebook for the selected `DATASET` / `MODEL` combo.

This notebook:
- Runs the vendored `PGExplainerExt` implementation through the shared benchmark pipeline.
- Loads the matching checkpoint and explain index for the chosen combo.
- Saves official run artifacts under `resources/results/official_pg/` and finishes with the shared one-spot metrics summary.

Usage:
- Change `DATASET` and `MODEL` in the first code cell when needed.
- Run the notebook top to bottom.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

_PROJECT_CANDIDATES = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
PROJECT_ROOT = next((p for p in _PROJECT_CANDIDATES if (p / "I_explainer_benchmark").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate project root from the current working directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from I_explainer_benchmark.src.notebooks.explainer_notebook_setup import (
    initialize_explainer_notebook,
    prepare_explainer_run,
)

# Select the dataset / model combo here.
DATASET = "simulate_v1"
MODEL = "tgn"

# Shared notebook setup. Leave the rest unchanged.
BOOT = initialize_explainer_notebook("06_pg.ipynb", dataset=DATASET, model=MODEL, start=PROJECT_ROOT)
NB = BOOT.env
CFG = BOOT.config
SETTINGS = BOOT.settings
BENCHMARK_ROOT = BOOT.bench_root
REPO_ROOT = BOOT.repo_root
TGN_VENDOR_ROOT = BOOT.paths["TGN_VENDOR_ROOT"]


In [2]:
import torch
from I_explainer_benchmark.src.notebooks.notebook_helpers import set_global_seed

MODEL = str(MODEL).strip().lower()
SUPPORTED_MODEL_TYPES = set(CFG["supported_model_types"])
if MODEL not in SUPPORTED_MODEL_TYPES:
    raise ValueError(f"Unsupported MODEL={MODEL!r}. Choose one of {sorted(SUPPORTED_MODEL_TYPES)}")

CTX = prepare_explainer_run("06_pg.ipynb", dataset=DATASET, model=MODEL, start=PROJECT_ROOT)

DATASET_NAME = CTX.dataset
MODEL_TYPE = CTX.model
SEED = int(CFG["seed"])
set_global_seed(SEED)

N_EVAL_EVENTS = int(SETTINGS["n_eval_events"])
EXPLAIN_START = int(SETTINGS.get("explain_start", 0))
CANDIDATES_SIZE = int(SETTINGS["candidates_size"])
USE_CACHED_PG_RESULTS = bool(SETTINGS["use_cached_pg_results"])


def _official_pg_train_epochs(dataset_name: str) -> int:
    per_dataset = dict(SETTINGS["pg_train_epochs_by_dataset"])
    default = int(SETTINGS["pg_train_epochs_default"])
    return int(per_dataset.get(str(dataset_name).lower(), default))


PG_TRAIN_EPOCHS = _official_pg_train_epochs(DATASET_NAME)
FLIP_PROGRESS_BUDGET = float(CFG["flip_progress_budget"])
FLIP_PROGRESS_AREA_THRESHOLD = float(CFG["flip_progress_area_threshold"])
FLIP_PROGRESS_POINT_THRESHOLD = float(CFG["flip_progress_point_threshold"])
TGNN_LEVELS = list(CFG["tgnn_levels"])
RANKING_CFG = dict(CFG["ranking_cfg"])
SOURCE_NOTEBOOK = str(CFG["source_notebook"])

EXPLAIN_INDEX_PATH = CTX.explain_index_path
CKPT_PATH = CTX.checkpoint_path
DEVICE = CTX.device
PG_CKPT_DIR = BENCHMARK_ROOT / "resources" / "models" / "pg" / DATASET_NAME / MODEL_TYPE
PG_CKPT_DIR.mkdir(parents=True, exist_ok=True)
out_dir = BENCHMARK_ROOT / "resources" / "results" / str(CFG["out_dir_name"])
out_dir.mkdir(parents=True, exist_ok=True)

print("DATASET:", DATASET_NAME)
print("MODEL  :", MODEL_TYPE)
print("CKPT   :", CKPT_PATH)
print("DEVICE :", DEVICE)


DATASET: simulate_v1
MODEL  : tgn
CKPT   : /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/models/simulate_v1/tgn/tgn_simulate_v1_best.pth
DEVICE : mps


In [3]:
# Run/reuse PG explanations and persist results.jsonl for downstream metrics
from pathlib import Path

import inspect
import numpy as np
import pandas as pd

from I_explainer_benchmark.src.explainers.adapters.pg_adapter import PGExplainerExt
from I_explainer_benchmark.src.explainers.builder import make_explainer_builder
from I_explainer_benchmark.src.notebooks.explainer_notebook_runtime import (
    load_jsonl_records,
    prepare_standard_edge_explainer_run,
    run_or_reuse_explanations,
)
from I_explainer_benchmark.src.notebooks.notebook_helpers import forced_target_event_ids_for_combo as _forced_target_event_ids_for_combo

# Hard-guard: ensure we are using the original PGExplainerExt from the tgnnexplainer submodule.
pg_src = Path(inspect.getfile(PGExplainerExt)).resolve()
pg_src_norm = str(pg_src).replace('\\', '/')
print('PG source:', pg_src)
assert '/submodules/explainer/tgnnexplainer/' in pg_src_norm, (
    'Expected PGExplainerExt implementation from submodules/explainer/tgnnexplainer'
)

FORCED_TARGET_EVENT_IDS = list(
    globals().get('FORCED_TARGET_EVENT_IDS', _forced_target_event_ids_for_combo(DATASET_NAME, MODEL_TYPE))
)

build_explainer = make_explainer_builder(
    dataset_name=DATASET_NAME,
    model_type=MODEL_TYPE,
    device=DEVICE,
    seed=SEED,
    callable_scope=globals(),
)

pg_explainer = build_explainer(
    'pg',
    allow_missing=False,
    overrides={
        'alias': 'pg',
        'explainer_name': 'pg_explainer_tg',
        'explanation_level': 'event',
        'threshold_num': 20,
        'train_epochs': int(PG_TRAIN_EPOCHS),
        'explainer_ckpt_dir': str(PG_CKPT_DIR),
        'reg_coefs': [0.5, 0.1],
        'batch_size': 16,
        'lr': 1e-4,
        'cache': True,
    },
)
print('Explainer alias:', pg_explainer.alias)

ctx = prepare_standard_edge_explainer_run(
    dataset_name=DATASET_NAME,
    model_name=MODEL_TYPE,
    ckpt_path=CKPT_PATH,
    device=DEVICE,
    explain_index_path=EXPLAIN_INDEX_PATH,
    n_eval_events=N_EVAL_EVENTS,
    out_dir=out_dir,
    explainer=pg_explainer,
    candidates_size=CANDIDATES_SIZE,
    seed=SEED,
    start=EXPLAIN_START,
    include_event_ids=FORCED_TARGET_EVENT_IDS,
)
model = ctx.model
events = ctx.events
backbone = model.backbone
all_event_idxs = ctx.all_event_idxs
target_event_idxs = ctx.target_event_idxs
anchors = ctx.anchors

print('Anchors selected:', len(anchors), '/', len(all_event_idxs))
if anchors:
    print('First target event_idx:', anchors[0]['event_idx'])

out, run_dir, out_jsonl, base_name = run_or_reuse_explanations(
    runner=ctx.runner,
    anchors=anchors,
    dataset_name=DATASET_NAME,
    model_name=MODEL_TYPE,
    explainer_name='pg',
    target_event_idxs=target_event_idxs,
    use_cached=USE_CACHED_PG_RESULTS,
)

print('run_dir:', run_dir)
print('out_jsonl:', out_jsonl)
print('base_name:', base_name)

_records = load_jsonl_records(out_jsonl)

pg_rows = []
for rec in _records:
    ctx_row = rec.get('context') or {}
    tgt = ctx_row.get('target') or {}
    res = rec.get('result') or {}
    extras = res.get('extras') or {}
    cand = list(extras.get('candidate_eidx') or [])
    imp = list(res.get('importance_edges') or [])
    pg_rows.append({
        'event_idx': int(tgt.get('event_idx')) if tgt.get('event_idx') is not None else pd.NA,
        'candidate_size': int(len(cand)),
        'importance_size': int(len(imp)),
        'importance_mean': float(np.mean(np.asarray(imp, dtype=float))) if imp else np.nan,
        'importance_std': float(np.std(np.asarray(imp, dtype=float))) if imp else np.nan,
    })

pg_results_df = pd.DataFrame(pg_rows)
out_csv = out_dir / f'{base_name}.csv'
pg_results_df.to_csv(out_csv, index=False)

print('Saved explanations:', out_csv)
pass  # suppressed intermediate display; one-spot summary is shown below


PG source: /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/submodules/explainer/tgnnexplainer/tgnnexplainer/xgraph/method/other_baselines_tg.py
Built explainer 'pg' from /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/configs/explainer/pg.json
Explainer alias: pg
102 events to explain
Anchors selected: 30 / 102
First target event_idx: 3


Evaluation:   0%|          | 0/30 [00:00<?, ?expl/s]

explainer ckpt loaded from /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/models/pg/simulate_v1/tgn/tgn_simulate_v1_pg_explainer_tg_expl_ckpt.pt

explain 0-th: 3
candidate scores saved at /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/results/candidate_scores/tgn_simulate_v1_pg_explainer_tg_3_candidate_scores_th20.csv

explain 0-th: 92
candidate scores saved at /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/results/candidate_scores/tgn_simulate_v1_pg_explainer_tg_92_candidate_scores_th20.csv

explain 0-th: 142
candidate scores saved at /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/results/candidate_scores/tgn_simulate_v1_pg_explainer_tg_142_candidate_scores_th20.csv

explain 0-th: 268
candidate scores saved at

In [4]:
# Shared metrics + export pipeline (clean notebook wrapper)
from I_explainer_benchmark.src.notebooks.notebook_metrics_pipeline import run_notebook_metrics_from_namespace

_pipeline_out = run_notebook_metrics_from_namespace(globals(), CFG)
globals().update(_pipeline_out)
run_dir_export = _pipeline_out['run_dir_export']
export_root = _pipeline_out['export_root']
out_jsonl = _pipeline_out['out_jsonl']
out_dir = _pipeline_out['out_dir']
base_name = _pipeline_out['base_name']
print('CURRENT_RUN_ID:', _pipeline_out['CURRENT_RUN_ID'])
print('Saved run export directory:', run_dir_export)
print('Updated global tables under:', export_root)


/Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/src/notebooks/notebook_runtime_common.py:39: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  prev = pd.read_csv(path)


CURRENT_RUN_ID: simulate_v1_tgn_official_pg_20260315_200703
Saved run export directory: /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/results/summary_ready/simulate_v1_tgn_official_pg_20260315_200703
Updated global tables under: /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/results/summary_ready


In [5]:
# One-spot metrics summary (official)
from I_explainer_benchmark.src.notebooks.notebook_display import show_one_spot_metrics_summary

_one_spot = globals().get("one_spot", globals().get("_pipeline_out", {}).get("one_spot", {}))
show_one_spot_metrics_summary(_one_spot)


One-spot metrics summary (official):


,dataset,model,explainer,variant,n_events,best_fid,best_fid_sparsity,aufsc,best_minus_aufsc,best_fid_raw,...,tempme_acc_auc.ratio_acc,tempme_acc_auc.ratio_prob,tempme_acc_auc.ratio_logit,tempme_acc_auc.ratio_aps,tempme_acc_auc.ratio_auc,temgx_aufsc,cody_AUFSC_plus,cody_AUFSC_minus,cody_CHARR,elapsed_sec
0,simulate_v1,tgn,pg,official,30,1.8049359009,1.0,1.4208156894,0.3841202114,1.8049359009,...,0.871875,0.0023930973,-0.0164159486,0.8291666667,0.75,0.4623504538,0.0153461705,0.0033710503,0.0055278199,0.0746878528
